# 13 — Chunking & Phrase Extraction

Chunking groups tokens into multi-word units that carry a single meaning: "deep learning expertise" is one chunk, not four unrelated words. Where Ch. 10 tagged parts of speech and Ch. 11 mapped dependencies, this chapter turns that structure into the phrase-level units that skills and job titles are actually made of.

**Why it matters for resumes / ATS:** skill phrases, job titles, and tool names are almost always multi-word ("machine learning", "computer vision", "Senior Data Scientist"). A keyword matcher that only sees single tokens either misses them or matches them spuriously. Chunking gives an ATS clean, complete phrases to index and match.

**Goal:** Extract meaningful multi-word phrases (noun chunks, verb phrases) from text.

A resume's signal lives in phrases, not isolated words. This chapter uses spaCy's dependency parse (Ch. 11) to pull out noun chunks (skills, titles) and verb phrases (actions + objects), then shows how phrase-level indexing beats token-level matching for skill lookup.

## 1. Noun Chunks with spaCy

A **noun chunk** is a noun plus its modifiers: "Senior data scientist" is one chunk rooted at *scientist*. spaCy builds these directly from the dependency tree via `doc.noun_chunks` — no grammar rules of your own.

**What the code does:** loads `en_core_web_sm`, parses a resume-style sentence, and prints each chunk with its root word and POS tag.
- `'Senior data scientist'` → root `scientist` (NOUN) — the title phrase, exactly as a recruiter would write it.
- `'strong Python skills'` → root `skills` (NOUN); `'deep learning expertise'` → root `expertise` (NOUN).

**Try it:** watch the chunk boundaries — "deep learning" stays together inside "deep learning expertise" instead of being split into separate tokens.

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
text = "Senior data scientist with strong Python skills and deep learning expertise"
doc = nlp(text)
for chunk in doc.noun_chunks:
    print(f"  '{chunk.text:35s}' root: {chunk.root.text:10s} ({chunk.root.pos_})")

  'Senior data scientist              ' root: scientist  (NOUN)
  'strong Python skills               ' root: skills     (NOUN)
  'deep learning expertise            ' root: expertise  (NOUN)


## 2. Extracting Skill Phrases from Resumes

Filtering chunks by their root's POS is a cheap, effective skill extractor: keep chunks whose root is a `NOUN` or `PROPN` (proper noun) and you keep skills and tools while dropping descriptive filler.

**What the code does:** `extract_phrases()` runs each sample resume through the pipeline and keeps only noun/proper-noun-rooted chunks.
- "Expert in Python, Machine Learning, and Deep Learning" → `['Expert', 'Python', 'Machine Learning', 'Deep Learning']` — multi-word skills preserved intact.
- "Proficient with TensorFlow, PyTorch, and Cloud Computing" → `['TensorFlow', 'PyTorch']` — "Cloud Computing" is dropped because its root's POS isn't NOUN/PROPN; a production extractor would tune this filter.

**Try it:** the second resume ("Natural Language Processing and Computer Vision") collapses into one long chunk — spaCy's coordination behavior across "and". Know this when you clean output.

In [2]:
resumes = [
    "Expert in Python, Machine Learning, and Deep Learning",
    "Skilled in Natural Language Processing and Computer Vision",
    "Proficient with TensorFlow, PyTorch, and Cloud Computing",
]
def extract_phrases(text):
    doc = nlp(text)
    return [chunk.text for chunk in doc.noun_chunks if chunk.root.pos_ in ("NOUN", "PROPN")]

for r in resumes:
    print(f"  '{r}'")
    print(f"    phrases: {extract_phrases(r)}")

  'Expert in Python, Machine Learning, and Deep Learning'
    phrases: ['Expert', 'Python', 'Machine Learning', 'Deep Learning']
  'Skilled in Natural Language Processing and Computer Vision'
    phrases: ['Natural Language Processing and Computer Vision']
  'Proficient with TensorFlow, PyTorch, and Cloud Computing'
    phrases: ['TensorFlow', 'PyTorch']


## 3. Verb Phrase (Action + Object) Extraction

Resume bullets are (action → object) pairs: "Developed ML models", "led teams". Extracting the verb and its direct object gives you the structured achievement unit from Ch. 11's SVO idea, at phrase level.

**What the code does:** `verb_phrases()` iterates tokens; for each `VERB` it looks for a `dobj` (direct object) and a `prep` (preposition) among the verb's children, returning `(verb, object, preposition)`.
- `"Developed ML models with TensorFlow and led teams"` → `[('Developed', None, None), ('led', 'teams', None)]`.

**Try it:** observe the honest failure mode — sentence-initial "Developed" is parsed as a modifier of "models", so it has no `dobj` child and yields `(None, None)`. Parsers are imperfect; production code should fall back to lemmas and accept participle readings.

In [3]:
def verb_phrases(text):
    doc = nlp(text)
    vps = []
    for token in doc:
        if token.pos_ == "VERB":
            obj = next((c.text for c in token.children if c.dep_ == "dobj"), None)
            prep = next((c.text for c in token.children if c.dep_ == "prep"), None)
            vps.append((token.text, obj, prep))
    return vps

text = "Developed ML models with TensorFlow and led teams"
print(f"  Verb phrases: {verb_phrases(text)}")

  Verb phrases: [('Developed', None, None), ('led', 'teams', None)]


## 4. Keyword Chunking for Resume Search

The payoff: a chunk-level index that matches skills as phrases. Instead of checking whether a single token appears, we check whether a known skill string appears inside any noun chunk — this catches "machine learning" inside "machine learning applications".

**What the code does:** `extract_skill_chunks()` loops over `doc.noun_chunks` and does a case-insensitive substring check against a `skills_db` set, returning `(skill, chunk)` pairs.
- With `skills = {"Python", "Machine Learning", "Deep Learning", "TensorFlow", "NLP"}` and the text "Expert in Python and Machine Learning applications", it returns `{('Python', 'Python'), ('Machine Learning', 'Machine Learning')}`.

**Try it:** "Deep Learning" and "NLP" are in the database but absent from the text, so they are correctly not reported — and the returned chunk is the *full* chunk, ready to be highlighted or indexed.

In [4]:
# Chunk-based phrase indexing
def extract_skill_chunks(text, skills_db):
    doc = nlp(text)
    found = set()
    for chunk in doc.noun_chunks:
        for skill in skills_db:
            if skill.lower() in chunk.text.lower():
                found.add((skill, chunk.text))
    return found

skills = {"Python", "Machine Learning", "Deep Learning", "TensorFlow", "NLP"}
print(extract_skill_chunks("Expert in Python and Machine Learning applications", skills))

{('Python', 'Python'), ('Machine Learning', 'Machine Learning')}


## Key Insight: Chunking preserves multi-word concepts that single-token approaches miss.

**Chunks are the atomic unit of a resume: skills, titles, and tools are phrases, so match at phrase level.**

Token-level features would shred "deep learning" into "deep" + "learning" and lose the concept; chunks keep it whole. Chunking is the bridge between linguistic structure (Ch. 11) and the vector representations that follow — the phrases extracted here are exactly the candidate terms that keyword extraction (Ch. 14) and TF-IDF/BoW features (Ch. 15–16) should operate on.